In [1]:
from importlib.metadata import version

print(version("langgraph"))

1.1.6


## 예제 1: 숫자 누적 계산기

In [2]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict

# ===== 1단계: 상태 정의 =====
# 그래프가 기억할 정보의 구조를 정의합니다
class CalculatorState(TypedDict):
    total: int        # 현재까지의 총합
    history: list     # 계산 기록

# ===== 2단계: 노드 함수 정의 =====
# 실제 계산을 수행하는 함수입니다
def add_number(state: CalculatorState) -> dict:
    """새 숫자를 더하고 기록을 남깁니다"""
    current_total = state["total"]
    number_to_add = 10  # 예제에서는 항상 10을 더함

    new_total = current_total + number_to_add
    new_history = state["history"] + [f"{current_total} + {number_to_add} = {new_total}"]

    print(f"📊 계산: {current_total} + {number_to_add} = {new_total}")

    return {
        "total": new_total,
        "history": new_history
    }

# ===== 3단계: 그래프 구성 =====
graph = StateGraph(CalculatorState)
graph.add_node("add", add_number)
graph.add_edge(START, "add")
graph.add_edge("add", END)

# ===== 4단계: 메모리 연결 =====
memory = InMemorySaver()
app = graph.compile(checkpointer=memory)

# ===== 5단계: 실행 =====
config = {"configurable": {"thread_id": "calculator_session"}}

# 첫 번째 계산 - 초기 상태 제공
print("=== 첫 번째 계산 ===")
result = app.invoke({"total": 0, "history": []}, config=config)
print(f"결과: total={result['total']}, history={result['history']}\n")

# 두 번째 계산 - 이전 상태 자동 복원!
print("=== 두 번째 계산 ===")
result = app.invoke({}, config=config)   # 모든 값 생략 가능!
print(f"결과: total={result['total']}, history={result['history']}\n")

# 세 번째 계산
print("=== 세 번째 계산 ===")
result = app.invoke({}, config=config)  
print(f"결과: total={result['total']}, history={result['history']}")


=== 첫 번째 계산 ===
📊 계산: 0 + 10 = 10
결과: total=10, history=['0 + 10 = 10']

=== 두 번째 계산 ===
📊 계산: 10 + 10 = 20
결과: total=20, history=['0 + 10 = 10', '10 + 10 = 20']

=== 세 번째 계산 ===
📊 계산: 20 + 10 = 30
결과: total=30, history=['0 + 10 = 10', '10 + 10 = 20', '20 + 10 = 30']


## 예제 2: 사용자별 방문 카운터

In [3]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict

# 상태 정의
class VisitorState(TypedDict):
    name: str           # 방문자 이름
    visit_count: int    # 방문 횟수
    message: str        # 환영 메시지

# 노드 함수: 방문자 맞이하기
def welcome_visitor(state: VisitorState) -> dict:
    """방문 횟수에 따라 다른 인사를 합니다"""
    name = state["name"]
    count = state["visit_count"] + 1

    # 방문 횟수에 따른 메시지 분기
    if count == 1:
        message = f"🎉 환영합니다, {name}님! 처음 오셨네요!"
    elif count < 5:
        message = f"👋 반갑습니다, {name}님! ({count}번째 방문)"
    else:
        message = f"⭐ {name}님, 단골이시네요! ({count}번째 방문)"

    return {"visit_count": count, "message": message}

# 그래프 구성
graph = StateGraph(VisitorState)
graph.add_node("welcome", welcome_visitor)
graph.add_edge(START, "welcome")
graph.add_edge("welcome", END)

# 메모리 설정
memory = InMemorySaver()
app = graph.compile(checkpointer=memory)

# ===== 서로 다른 사용자 시뮬레이션 =====

# Alice 설정 (thread_id로 구분)
alice_config = {"configurable": {"thread_id": "user_alice"}}

# Bob 설정
bob_config = {"configurable": {"thread_id": "user_bob"}}

print("=" * 50)
print("Alice 첫 방문")
result = app.invoke({"name": "Alice", "visit_count": 0, "message": ""}, config=alice_config)
print(result["message"])

print("\n" + "=" * 50)
print("Bob 첫 방문")
result = app.invoke({"name": "Bob", "visit_count": 0, "message": ""}, config=bob_config)
print(result["message"])

print("\n" + "=" * 50)
print("Alice 두 번째 방문")
# name만 전달 - visit_count는 메모리에서 자동 복원!
result = app.invoke({"name": "Alice", "message": ""}, config=alice_config)
print(result["message"])

print("\n" + "=" * 50)
print("Alice 세 번째 ~ 다섯 번째 방문")
for i in range(3):
    result = app.invoke({"name": "Alice", "message": ""}, config=alice_config)
    print(result["message"])


Alice 첫 방문
🎉 환영합니다, Alice님! 처음 오셨네요!

Bob 첫 방문
🎉 환영합니다, Bob님! 처음 오셨네요!

Alice 두 번째 방문
👋 반갑습니다, Alice님! (2번째 방문)

Alice 세 번째 ~ 다섯 번째 방문
👋 반갑습니다, Alice님! (3번째 방문)
👋 반갑습니다, Alice님! (4번째 방문)
⭐ Alice님, 단골이시네요! (5번째 방문)


## 예제 3: 대화 기록을 저장하는 챗봇

In [4]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict, Annotated
from datetime import datetime


# 상태 정의 - add_messages reducer 추가
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]  # ← reducer로 자동 누적!
    user_input: str


def process_message(state: ChatState) -> dict:
    """사용자 메시지를 받아 응답을 생성합니다"""
    user_input = state["user_input"]
    messages = state.get("messages", [])  # 기존 메시지 읽기

    # 사용자 메시지
    user_msg = {
        "role": "user",
        "content": user_input,
        "time": datetime.now().strftime("%H:%M")
    }

    # 응답 생성
    if "안녕" in user_input:
        response = "안녕하세요! 무엇을 도와드릴까요?"
    elif "이름" in user_input:
        response = "저는 LangGraph 챗봇입니다!"
    elif "기억" in user_input or "대화" in user_input:
        # 기존 메시지에서 user 메시지 수 + 현재 메시지
        history_count = len([m for m in messages if getattr(m, 'type', None) == 'human' or (isinstance(m, dict) and m.get("role") == "user")]) + 1
        response = f"네, 지금까지 {history_count}번의 대화를 나눴습니다!"
    else:
        response = f"'{user_input}'에 대해 말씀해 주셨네요."

    assistant_msg = {
        "role": "assistant", 
        "content": response,
        "time": datetime.now().strftime("%H:%M")
    }

    # add_messages reducer가 자동으로 누적시킴
    return {"messages": [user_msg, assistant_msg]}


# 그래프 구성
graph = StateGraph(ChatState)
graph.add_node("chat", process_message)
graph.add_edge(START, "chat")
graph.add_edge("chat", END)

memory = InMemorySaver()
app = graph.compile(checkpointer=memory)

session_config = {"configurable": {"thread_id": "chat_session_001"}}


def chat(user_message: str) -> str:
    """사용자 메시지를 보내고 응답을 받습니다"""
    result = app.invoke(
        {"user_input": user_message}, 
        config=session_config
    )
    return result["messages"][-1].content

# 대화 시뮬레이션
print("🤖 챗봇: 대화를 시작합니다!\n")

print(f"👤 사용자: 안녕!")
print(f"🤖 챗봇: {chat('안녕!')}\n")

print(f"👤 사용자: 네 이름이 뭐야?")
print(f"🤖 챗봇: {chat('네 이름이 뭐야?')}\n")

print(f"👤 사용자: 오늘 날씨 좋다")
print(f"🤖 챗봇: {chat('오늘 날씨 좋다')}\n")

print(f"👤 사용자: 우리 몇 번 대화했지?")
print(f"🤖 챗봇: {chat('우리 몇 번 대화했지?')}\n")

# 전체 대화 기록 확인
print("=" * 50)
print("📝 전체 대화 기록:")
state = app.get_state(session_config)
for msg in state.values["messages"]:
    role = "👤" if msg.type == "human" else "🤖"
    print(f"  {role} {msg.content}")


🤖 챗봇: 대화를 시작합니다!

👤 사용자: 안녕!
🤖 챗봇: 안녕하세요! 무엇을 도와드릴까요?

👤 사용자: 네 이름이 뭐야?
🤖 챗봇: 저는 LangGraph 챗봇입니다!

👤 사용자: 오늘 날씨 좋다
🤖 챗봇: '오늘 날씨 좋다'에 대해 말씀해 주셨네요.

👤 사용자: 우리 몇 번 대화했지?
🤖 챗봇: 네, 지금까지 4번의 대화를 나눴습니다!

📝 전체 대화 기록:
  👤 안녕!
  🤖 안녕하세요! 무엇을 도와드릴까요?
  👤 네 이름이 뭐야?
  🤖 저는 LangGraph 챗봇입니다!
  👤 오늘 날씨 좋다
  🤖 '오늘 날씨 좋다'에 대해 말씀해 주셨네요.
  👤 우리 몇 번 대화했지?
  🤖 네, 지금까지 4번의 대화를 나눴습니다!


## create_agent로 간편하게 메모리 사용하기

In [5]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

# LLM 초기화
llm = init_chat_model("openai:gpt-4.1-mini")

# 에이전트 생성 (checkpointer로 메모리 자동 연결)
agent = create_agent(
    model=llm,
    tools=[],
    checkpointer=InMemorySaver(),
    system_prompt="당신은 친절한 한국어 챗봇입니다.",
)

# 실행 - thread_id로 대화 세션 구분
config = {"configurable": {"thread_id": "user_123"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "안녕! 내 이름은 철수야"}]},
    config=config,
)
print(response["messages"][-1].content)

# 이전 대화를 기억!
response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐였지?"}]},
    config=config,
)
print(response["messages"][-1].content)  # "철수님"이라고 응답


안녕 철수야! 만나서 반가워. 오늘 기분은 어때?
너의 이름은 철수야. 기억하고 있어! 다른 궁금한 거 있어?


## 상태 검사

In [7]:
from pprint import pprint

# 특정 thread의 현재 상태 가져오기
state = agent.get_state(config)

# 저장된 값 출력
print("=== 저장된 상태 값 ===")
pprint(state.values)

# 설정 정보 출력
print("\n=== 설정 정보 ===")
pprint(state.config)

# 메시지 개수 확인
print(f"\n총 메시지 수: {len(state.values['messages'])}")


=== 저장된 상태 값 ===
{'messages': [HumanMessage(content='안녕! 내 이름은 철수야', additional_kwargs={}, response_metadata={}, id='48bd9d14-62cf-4d33-a4d4-eef410f8293b'),
              AIMessage(content='안녕 철수야! 만나서 반가워. 오늘 기분은 어때?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 34, 'total_tokens': 53, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_23a909fd03', 'id': 'chatcmpl-DSYgsYvZZtKZfgHneKDcj9LsGuu9J', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d6fe7-05d5-7dc1-9561-2ab0a9b70223-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 34, 'output_tokens': 19, 'total_tokens': 53, 'input_token_details': {'audio': 0, 'cach

In [8]:
from rich.pretty import pprint

# 특정 thread의 현재 상태 가져오기
state = agent.get_state(config)

# 저장된 값 출력
print("=== 저장된 상태 값 ===")
pprint(state.values)

# 설정 정보 출력
print("\n=== 설정 정보 ===")
pprint(state.config)

# 메시지 개수 확인
print(f"\n총 메시지 수: {len(state.values['messages'])}")

=== 저장된 상태 값 ===


{
│   'messages': [
│   │   HumanMessage(
│   │   │   content='안녕! 내 이름은 철수야',
│   │   │   additional_kwargs={},
│   │   │   response_metadata={},
│   │   │   id='48bd9d14-62cf-4d33-a4d4-eef410f8293b'
│   │   ),
│   │   AIMessage(
│   │   │   content='안녕 철수야! 만나서 반가워. 오늘 기분은 어때?',
│   │   │   additional_kwargs={'refusal': None},
│   │   │   response_metadata={
│   │   │   │   'token_usage': {
│   │   │   │   │   'completion_tokens': 19,
│   │   │   │   │   'prompt_tokens': 34,
│   │   │   │   │   'total_tokens': 53,
│   │   │   │   │   'completion_tokens_details': {
│   │   │   │   │   │   'accepted_prediction_tokens': 0,
│   │   │   │   │   │   'audio_tokens': 0,
│   │   │   │   │   │   'reasoning_tokens': 0,
│   │   │   │   │   │   'rejected_prediction_tokens': 0
│   │   │   │   │   },
│   │   │   │   │   'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
│   │   │   │   },
│   │   │   │   'model_provider': 'openai',
│   │   │   │   'model_name': 'gpt-4.1-mini-2025-04-14',
│   │   │   │   'system_fingerprint': 'fp_23a909fd03',
│   │   │   │   'id': 'chatcmpl-DSYgsYvZZtKZfgHneKDcj9LsGuu9J',
│   │   │   │   'service_tier': 'default',
│   │   │   │   'finish_reason': 'stop',
│   │   │   │   'logprobs': None
│   │   │   },
│   │   │   id='lc_run--019d6fe7-05d5-7dc1-9561-2ab0a9b70223-0',
│   │   │   tool_calls=[],
│   │   │   invalid_tool_calls=[],
│   │   │   usage_metadata={
│   │   │   │   'input_tokens': 34,
│   │   │   │   'output_tokens': 19,
│   │   │   │   'total_tokens': 53,
│   │   │   │   'input_token_details': {'audio': 0, 'cache_read': 0},
│   │   │   │   'output_token_details': {'audio': 0, 'reasoning': 0}
│   │   │   }
│   │   ),
│   │   HumanMessage(
│   │   │   content='내 이름이 뭐였지?',
│   │   │   additional_kwargs={},
│   │   │   response_metadata={},
│   │   │   id='5c156a10-b7dd-45be-9715-ca39aee3c7e2'
│   │   ),
│   │   AIMessage(
│   │   │   content='너의 이름은 철수야. 기억하고 있어! 다른 궁금한 거 있어?',
│   │   │   additional_kwargs={'refusal': None},
│   │   │   response_metadata={
│   │   │   │   'token_usage': {
│   │   │   │   │   'completion_tokens': 19,
│   │   │   │   │   'prompt_tokens': 68,
│   │   │   │   │   'total_tokens': 87,
│   │   │   │   │   'completion_tokens_details': {
│   │   │   │   │   │   'accepted_prediction_tokens': 0,
│   │   │   │   │   │   'audio_tokens': 0,
│   │   │   │   │   │   'reasoning_tokens': 0,
│   │   │   │   │   │   'rejected_prediction_tokens': 0
│   │   │   │   │   },
│   │   │   │   │   'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
│   │   │   │   },
│   │   │   │   'model_provider': 'openai',
│   │   │   │   'model_name': 'gpt-4.1-mini-2025-04-14',
│   │   │   │   'system_fingerprint': 'fp_d9b09842cd',
│   │   │   │   'id': 'chatcmpl-DSYgtPEjsMvpYmCfxKHvIREpDV0wO',
│   │   │   │   'service_tier': 'default',
│   │   │   │   'finish_reason': 'stop',
│   │   │   │   'logprobs': None
│   │   │   },
│   │   │   id='lc_run--019d6fe7-0d26-7be3-b314-525f2950b0d6-0',
│   │   │   tool_calls=[],
│   │   │   invalid_tool_calls=[],
│   │   │   usage_metadata={
│   │   │   │   'input_tokens': 68,
│   │   │   │   'output_tokens': 19,
│   │   │   │   'total_tokens': 87,
│   │   │   │   'input_token_details': {'audio': 0, 'cache_read': 0},
│   │   │   │   'output_token_details': {'audio': 0, 'reasoning': 0}
│   │   │   }
│   │   )
│   ]
}


=== 설정 정보 ===


{
│   'configurable': {
│   │   'thread_id': 'user_123',
│   │   'checkpoint_ns': '',
│   │   'checkpoint_id': '1f133b54-6469-630c-8004-ce2726d77641'
│   }
}


총 메시지 수: 4
